# 04 — Nettoyage et préparation des données

Ce notebook applique, **variable par variable**, les décisions de nettoyage issues du rapport de qualité produit dans `03_controle_qualite.ipynb` (`outputs/reports/rapport_qualite_donnees_brutes.md`).

Règles suivies :

1. Chaque problème est d'abord **identifié** (rappel du constat qualité).
2. Une **stratégie** est proposée.
3. La stratégie est **justifiée** (raison statistique ou métier) — jamais choisie par défaut.
4. Le traitement est **appliqué** via les fonctions réutilisables de `src/data/cleaning.py`.
5. Un **avant/après** est produit pour vérifier l'effet réel du traitement.

Toute la logique de transformation vit dans `src/data/cleaning.py` : ce notebook ne fait qu'orchestrer et documenter.

Les données brutes (`data/raw/`) ne sont **jamais modifiées**. Les résultats sont sauvegardés dans `data/interim/` puis `data/processed/`.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import (
    load_raw_dataset,
    TransformationJournal,
    remove_duplicate_rows,
    fix_column_types,
    harmonize_categories,
    handle_impossible_values,
    analyze_missing_values,
    impute_missing_values,
    compare_before_after,
    save_interim_dataset,
    save_processed_dataset,
)
from src.utils.paths import REPORTS_DIR

pd.set_option("display.max_columns", 40)

result = load_raw_dataset()
df_raw = result.dataframe
print("Fichier source :", result.path)
print("Dimensions brutes :", df_raw.shape)
df_raw.head()

Fichier source : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\data\raw\donnees_brutes_education_prioritaire.csv
Dimensions brutes : (5250, 22)


,annee,ecole_id,academie,departement,niveau,statut,rep,rep_plus,education_prioritaire,effectif_eleves,nombre_classes,taille_moyenne_classe,dedoublement,ips,score_francais,score_mathematiques,score_global,taux_maitrise_francais,taux_maitrise_mathematiques,variable_cible,source,source_url
0,2017,1,Grenoble,Puy-de-Dôme,CP,REP,1,0,1,16,2,18.73,0,44.69,52.48,50.99,51.74,53.47,52.61,51.74,simulation_reproductible,https://www.education.gouv.fr/depp/taille-des-...
1,2017,1,Grenoble,Puy-de-Dôme,CE1,REP,1,0,1,16,2,18.73,0,44.69,61.50,63.02,62.26,64.16,61.12,62.26,simulation_reproductible,https://www.education.gouv.fr/depp/taille-des-...
2,2017,1,Grenoble,Puy-de-Dôme,CM1,REP,1,0,1,16,2,18.73,0,44.69,77.42,77.29,77.35,68.75,72.03,77.35,simulation_reproductible,https://www.education.gouv.fr/depp/taille-des-...
3,2017,1,Grenoble,Puy-de-Dôme,CM2,REP,1,0,1,16,2,18.73,0,44.69,73.03,74.54,73.78,68.92,67.95,73.78,simulation_reproductible,https://www.education.gouv.fr/depp/taille-des-...
4,2017,1,Grenoble,Puy-de-Dôme,6e,REP,1,0,1,16,2,18.73,0,44.69,80.28,83.17,81.72,73.58,72.84,81.72,simulation_reproductible,https://www.education.gouv.fr/depp/taille-des-...


## Journal des transformations

Toutes les étapes appliquées ci-dessous sont enregistrées dans un journal unique, afin de garantir la traçabilité complète du nettoyage.

In [2]:
journal = TransformationJournal(dataset_name="donnees_education_prioritaire")
df = df_raw.copy()

## 1. Doublons

**Constat qualité (phase 7)** : `DUPLICATE_ROWS` = 0 ligne concernée, `DUPLICATE_IDENTIFIERS` (annee, ecole_id, niveau) = 0 combinaison dupliquée.

**Stratégie proposée** : aucune suppression n'est nécessaire, mais le contrôle est exécuté explicitement afin que la vérification reste tracée dans le journal (et se déclenche automatiquement si une future version des données brutes contient des doublons).

In [3]:
df = remove_duplicate_rows(df, journal)
print("Dimensions après contrôle des doublons :", df.shape)

Dimensions après contrôle des doublons : (5250, 22)


## 2. Types incohérents

**Constat qualité** : le contrôle qualité (`check_type_consistency`) n'a signalé aucune incohérence de type sur ce jeu de données. On documente cependant explicitement le typage attendu, pour figer un contrat de schéma robuste avant les étapes d'analyse :

- `annee` → entier ;
- `niveau`, `statut`, `academie`, `departement` → catégoriel (nombre de modalités limité, réutilisé dans les groupements) ;
- les indicateurs binaires (`rep`, `rep_plus`, `education_prioritaire`, `dedoublement`) restent des entiers 0/1 (plus pratiques pour les régressions que des booléens).

**Justification** : fixer un typage explicite évite les erreurs silencieuses lors des jointures et des group-by dans les notebooks suivants (05 à 09).

In [4]:
type_cible = {
    "annee": "int64",
    "niveau": "category",
    "statut": "category",
    "academie": "category",
    "departement": "category",
}
df = fix_column_types(df, type_cible, journal)
df.dtypes

annee                             int64
ecole_id                          int64
academie                       category
departement                    category
niveau                         category
statut                         category
rep                               int64
rep_plus                          int64
education_prioritaire             int64
effectif_eleves                   int64
nombre_classes                    int64
taille_moyenne_classe           float64
dedoublement                      int64
ips                             float64
score_francais                  float64
score_mathematiques             float64
score_global                    float64
taux_maitrise_francais          float64
taux_maitrise_mathematiques     float64
variable_cible                  float64
source                              str
source_url                          str
dtype: object

## 3. Catégories inattendues

**Constat qualité** : `check_unexpected_categories` n'a détecté aucune modalité hors référentiel. Les valeurs observées sont conformes :

- `statut` : `REP`, `REP+`, `Hors EP` ;
- `niveau` : `CP`, `CE1`, `CM1`, `CM2`, `6e`.

**Stratégie** : harmonisation appliquée par sécurité (suppression d'espaces parasites, casse) même si aucune anomalie n'a été trouvée — coût nul, robustesse accrue si une prochaine extraction Open Data introduit des variantes d'écriture (ex. `REP +`, `Hors-EP`).

In [5]:
# harmonize_categories attend, par colonne, un mapping {valeur_observee: valeur_canonique}.
# Ici les modalites observees sont deja conformes : le mapping est "identite", ce qui documente
# explicitement le referentiel attendu sans rien modifier si aucune variante n'est trouvee.
mapping_statut = {v: v for v in ["REP", "REP+", "Hors EP"]}
mapping_niveau = {v: v for v in ["CP", "CE1", "CM1", "CM2", "6e"]}
df = harmonize_categories(df, "statut", mapping_statut, journal)
df = harmonize_categories(df, "niveau", mapping_niveau, journal)
for col in ["statut", "niveau"]:
    print(col, ":", sorted(df[col].astype(str).unique()))

statut : ['Hors EP', 'REP', 'REP+']
niveau : ['6e', 'CE1', 'CM1', 'CM2', 'CP']


## 4. Valeurs impossibles

**Constat qualité** : `check_impossible_values` n'a signalé aucune valeur hors plage plausible sur les variables bornées (`ips` ∈ [0, 160], scores et taux ∈ [0, 100], `effectif_eleves` > 0, `taille_moyenne_classe` > 0).

**Stratégie** : on exécute quand même le contrôle explicitement (traçabilité) sur les variables les plus sensibles. Si une valeur impossible apparaissait (ex. `ips` négatif, taux > 100 %), la stratégie retenue serait de la **marquer manquante** (`marquer_manquant`) plutôt que de la supprimer : cela évite de perdre les autres variables de la même ligne, et permet un traitement explicite lors de l'étape d'imputation.

In [6]:
df = handle_impossible_values(df, "ips", (0, 160), "marquer_manquant", journal)
df = handle_impossible_values(df, "effectif_eleves", (1, 60), "marquer_manquant", journal)
df = handle_impossible_values(df, "taille_moyenne_classe", (1, 40), "marquer_manquant", journal)
for col in ["score_francais", "score_mathematiques", "score_global", "taux_maitrise_francais", "taux_maitrise_mathematiques", "variable_cible"]:
    df = handle_impossible_values(df, col, (0, 100), "marquer_manquant", journal)
print("Valeurs manquantes introduites par ce contrôle :", df.isna().sum().sum())

Valeurs manquantes introduites par ce contrôle : 0


**Résultat** : 0 valeur manquante introduite — confirmant qu'aucune valeur impossible n'existait réellement dans les données brutes. Cette étape reste utile car elle sera automatiquement réactivée si une version future des données contient des valeurs hors plage.

## 5. Anomalie IMPORTANTE : `nombre_classes` à variance nulle

**Constat qualité** : `ZERO_VARIANCE` sur `nombre_classes` — la variable ne prend qu'une seule valeur (`2.0`) sur les 5250 observations.

**Diagnostic** : en inspectant la fonction de génération (`src/data/acquisition.py`, `build_synthetic_open_data`), la formule `nombre_classes = max(2, round(effectif_ecole / (18 + alea)))` retombe systématiquement sur la borne minimale `2` pour les effectifs d'école simulés. La variable est donc **non informative** dans ce jeu de données : elle ne peut apporter aucune capacité explicative (variance nulle ⇒ corrélation nulle avec toute autre variable, coefficient de régression non identifiable).

**Stratégie retenue** : **exclusion documentée de la variable** dans le jeu de données prêt pour l'analyse (`data/processed/`), plutôt qu'une imputation ou une correction arbitraire de sa valeur (qui reviendrait à fabriquer une donnée inexistante). La variable reste présente dans `data/interim/` (traçabilité), mais est retirée avant modélisation.

**Justification** : 
- Statistiquement, une variable à variance nulle n'apporte aucune information et peut faire échouer certains algorithmes (division par zéro dans la standardisation, colinéarité parfaite).
- Métier : l'information utile pour capturer l'effet du dédoublement est déjà portée par `taille_moyenne_classe` et par l'indicateur binaire `dedoublement`, qui varient correctement dans les données. `nombre_classes` est donc redondant, pas remplacé.
- Alternative écartée : "corriger" `nombre_classes` en le recalculant à partir de `effectif_eleves` reviendrait à fabriquer une donnée non observée dans la source d'origine — contraire à la consigne de ne jamais fabriquer de résultat ou de valeur non justifiée.

In [7]:
from src.data.cleaning import TransformationStep

journal.log(
    TransformationStep(
        action="exclusion_variable",
        variable="nombre_classes",
        strategie="exclusion_de_la_variable",
        justification=(
            "Variance nulle confirmée par le contrôle qualité (phase 7) : la variable ne prend "
            "qu'une seule valeur (2.0) sur 5250 observations. Aucune capacité explicative possible. "
            "L'information pertinente est déjà portée par 'taille_moyenne_classe' et 'dedoublement'. "
            "Conservée dans data/interim/ pour traçabilité, retirée de data/processed/ (jeu prêt pour analyse)."
        ),
        n_lignes_avant=len(df),
        n_lignes_apres=len(df),
        n_valeurs_modifiees=0,
    )
)
print("Décision enregistrée dans le journal.")

Décision enregistrée dans le journal.


## 6. Valeurs manquantes

**Constat qualité** : `check_missing_values` ne signale **aucune valeur manquante** dans les données brutes (0 NA sur 22 colonnes × 5250 lignes).

In [8]:
rapport_na = analyze_missing_values(df)
rapport_na

,n_manquants,pourcentage_manquant
annee,0,0.0
ecole_id,0,0.0
source,0,0.0
variable_cible,0,0.0
taux_maitrise_mathematiques,0,0.0
taux_maitrise_francais,0,0.0
score_global,0,0.0
score_mathematiques,0,0.0
score_francais,0,0.0
ips,0,0.0


**Stratégie retenue** : **aucune imputation** n'est nécessaire ni appliquée, puisqu'il n'existe aucune valeur manquante réelle à ce stade. 

Le module `src/data/cleaning.py` fournit néanmoins une fonction générique `impute_missing_values(df, colonne, strategie, justification, ...)` supportant explicitement : `suppression`, `moyenne`, `mediane`, `mode`, `categorie_inconnu`, `imputation_par_groupe`, `knn`, `mice`. 

Elle **n'autorise aucun choix par défaut** : la stratégie doit être fournie explicitement par l'appelant, sous peine de `ValueError`. Cette fonction sera réutilisée sans modification si de futures extractions Open Data (données réelles DEPP) contiennent des valeurs manquantes — ce qui est probable, notamment pour les niveaux/années non couverts par certaines sources. 

À titre de démonstration du bon fonctionnement (sans altérer les données réelles), la cellule suivante illustre l'utilisation de la fonction sur une copie temporaire, puis n'est **pas** appliquée au jeu de données final.

In [9]:
import numpy as np

demo = df.copy()
demo.loc[demo.sample(5, random_state=42).index, "ips"] = np.nan
journal_demo = TransformationJournal(dataset_name="demo_imputation_ips")
demo_impute = impute_missing_values(
    demo,
    "ips",
    "imputation_par_groupe",
    justification=(
        "Démonstration : l'IPS dépend fortement du statut REP/REP+/Hors EP et de l'académie ; "
        "une imputation par groupe (moyenne par statut x académie) est plus fidèle qu'une moyenne globale."
    ),
    journal=journal_demo,
    group_columns=["statut", "academie"],
)
print("NA avant :", demo["ips"].isna().sum(), "| NA après (démo) :", demo_impute["ips"].isna().sum())
print("Cette démonstration n'affecte pas le jeu de données final (df).")

NA avant : 5 | NA après (démo) : 0
Cette démonstration n'affecte pas le jeu de données final (df).


## 7. Valeurs aberrantes (outliers IQR)

**Constat qualité** : la méthode IQR détecte des valeurs extrêmes (classées MINEUR) sur `effectif_eleves`, `taille_moyenne_classe`, `score_francais`, `score_mathematiques`, `score_global`, `taux_maitrise_francais`, `taux_maitrise_mathematiques`, `variable_cible` — entre 0.13 % et 0.31 % des observations selon la variable.

**Analyse** : ces variables sont toutes bornées par construction (scores et taux entre 0 et 100, effectifs positifs et plafonnés). Les valeurs signalées comme "aberrantes" par la méthode IQR restent **dans les plages plausibles métier** (aucune n'a été signalée comme valeur impossible à l'étape 4). Elles correspondent à la queue de distribution naturelle (écoles à faible ou fort effectif, élèves en grande difficulté ou en réussite exceptionnelle) plutôt qu'à des erreurs de saisie.

**Stratégie retenue** : **conservation** de ces observations, sans suppression ni plafonnement (winsorisation). 

**Justification** :
- Supprimer ces lignes réduirait artificiellement la variance et biaiserait l'estimation de l'effet du dédoublement, en particulier pour les analyses en sous-groupes (REP+, DROM, écoles à faible effectif) qui sont justement les cas d'intérêt du projet.
- Une winsorisation (plafonnement) modifierait des valeurs réelles sans justification empirique claire ; la consigne du projet interdit toute correction non justifiée.
- Ces observations seront réexaminées, si besoin, au moment de la modélisation (notebook 06), où des méthodes robustes (régression robuste, retrait ponctuel documenté) pourront être envisagées **avec justification statistique spécifique au modèle utilisé**, ce qui sort du périmètre du nettoyage général.

In [10]:
journal.log(
    TransformationStep(
        action="revue_outliers_iqr",
        variable="effectif_eleves, taille_moyenne_classe, score_francais, score_mathematiques, score_global, taux_maitrise_francais, taux_maitrise_mathematiques, variable_cible",
        strategie="conservation_sans_modification",
        justification=(
            "Valeurs aberrantes IQR (classées MINEUR en phase 7) toutes situées dans les plages "
            "plausibles métier (0-100 pour les scores/taux, effectifs positifs). Représentent une "
            "variation naturelle et non des erreurs de saisie. Leur suppression biaiserait l'estimation "
            "de l'effet du dédoublement sur les sous-groupes d'intérêt (REP+, petites écoles)."
        ),
        n_lignes_avant=len(df),
        n_lignes_apres=len(df),
        n_valeurs_modifiees=0,
    )
)
print("Décision de conservation des outliers enregistrée dans le journal.")

Décision de conservation des outliers enregistrée dans le journal.

## 8. Cohérence entre variables

**Constat qualité** : `check_cross_variable_consistency` n'a signalé aucune incohérence (statut ↔ rep/rep_plus, education_prioritaire, cohérence des tailles de classe, `score_global` ↔ moyenne des scores par matière).

**Stratégie** : aucune correction nécessaire ; contrôle simplement journalisé pour mémoire.

In [11]:
journal.log(
    TransformationStep(
        action="controle_coherence_croisee",
        variable="statut, rep, rep_plus, education_prioritaire, score_global",
        strategie="aucune_action",
        justification="Aucune incohérence détectée par le contrôle qualité (phase 7) ; aucune correction nécessaire.",
        n_lignes_avant=len(df),
        n_lignes_apres=len(df),
        n_valeurs_modifiees=0,
    )
)

## 9. Comparaison avant / après

In [12]:
colonnes_numeriques = [
    "effectif_eleves",
    "taille_moyenne_classe",
    "ips",
    "score_francais",
    "score_mathematiques",
    "score_global",
    "variable_cible",
]
comparaison = compare_before_after(df_raw, df, columns=colonnes_numeriques)
comparaison

,variable,n_avant,n_apres,moyenne_avant,moyenne_apres,mediane_avant,mediane_apres,ecart_type_avant,ecart_type_apres,n_manquants_avant,n_manquants_apres
0,effectif_eleves,5250,5250,20.060,20.060,20.000,20.000,5.905,5.905,0,0
1,taille_moyenne_classe,5250,5250,16.863,16.863,16.395,16.395,4.070,4.070,0,0
2,ips,5250,5250,72.804,72.804,73.355,73.355,13.501,13.501,0,0
3,score_francais,5250,5250,79.159,79.159,78.895,78.895,10.302,10.302,0,0
4,score_mathematiques,5250,5250,77.304,77.304,77.045,77.045,11.004,11.004,0,0
5,score_global,5250,5250,78.231,78.231,77.985,77.985,10.474,10.474,0,0
6,variable_cible,5250,5250,78.231,78.231,77.985,77.985,10.474,10.474,0,0


**Interprétation** : les statistiques descriptives (moyenne, écart-type, min, max) sont strictement identiques avant et après nettoyage pour toutes les variables numériques d'intérêt. C'est le résultat attendu : le contrôle qualité n'a révélé aucune anomalie nécessitant une correction de valeur — les seules décisions prises ont été (1) le typage explicite, (2) l'harmonisation défensive des catégories, et (3) l'exclusion documentée de `nombre_classes`. Aucune valeur n'a été fabriquée ni supprimée.

## 10. Sauvegarde des jeux de données

- `data/interim/` : jeu nettoyé et typé, avec `nombre_classes` encore présent (traçabilité complète).
- `data/processed/` : jeu prêt pour l'analyse, `nombre_classes` retiré (variable non informative, cf. étape 5).

In [13]:
chemin_interim = save_interim_dataset(df, "education_prioritaire")
print("Sauvegardé (interim) :", chemin_interim)

df_processed = df.drop(columns=["nombre_classes"])
chemin_processed = save_processed_dataset(df_processed, "education_prioritaire")
print("Sauvegardé (processed) :", chemin_processed)
print("Dimensions data/processed :", df_processed.shape)

Sauvegardé (interim) : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\data\interim\education_prioritaire_clean.csv


Sauvegardé (processed) : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\data\processed\education_prioritaire_analysis_ready.csv
Dimensions data/processed : (5250, 21)


## 11. Journal des transformations

Le journal complet (markdown + JSON) est sauvegardé dans `outputs/reports/`, au même emplacement que le rapport de qualité de la phase précédente, pour assurer la continuité de la documentation du projet.

In [14]:
chemins_journal = journal.save(REPORTS_DIR, "journal_transformations_donnees_education_prioritaire")
print("Journal sauvegardé :")
for kind, p in chemins_journal.items():
    print(" -", kind, ":", p)
print()
print(journal.to_markdown())

Journal sauvegardé :
 - markdown : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\outputs\reports\journal_transformations_donnees_education_prioritaire.md
 - json : C:\Users\admin\Desktop\PROJET STATISTIQUE PUBLIC B\outputs\reports\journal_transformations_donnees_education_prioritaire.json

# Journal des transformations — donnees_education_prioritaire

Nombre d'étapes appliquées : 19

## Étape 1 — suppression_doublons

- Stratégie retenue : **aucune_action**
- Justification : Aucun doublon strict détecté ; aucune suppression nécessaire.
- Lignes avant : 5250 — Lignes après : 5250
- Valeurs modifiées / concernées : 0

## Étape 2 — correction_type (variable : niveau)

- Stratégie retenue : **conversion_type**
- Justification : Le type observé ('str') ne correspondait pas au type attendu ('category') pour cette variable ; conversion appliquée pour garantir la cohérence des traitements statistiques ultérieurs.
- Lignes avant : 5250 — Lignes après : 5250
- Valeurs modifiées / concernées

## Synthèse du nettoyage

| Problème (phase 7) | Décision | Justification résumée |
|---|---|---|
| Doublons (0 trouvé) | Aucune action | Contrôle exécuté, rien à corriger |
| Types | Typage explicite (`category`, `int64`) | Fiabiliser les group-by / jointures futures |
| Catégories inattendues (0 trouvée) | Harmonisation défensive | Robustesse si futures données réelles moins propres |
| Valeurs impossibles (0 trouvée) | Contrôle actif, 0 valeur modifiée | Garde-fou pour données réelles futures |
| `nombre_classes` à variance nulle (IMPORTANT) | Exclusion de `data/processed/`, conservée dans `data/interim/` | Variable non informative ; information déjà portée par `taille_moyenne_classe` et `dedoublement` |
| Valeurs manquantes (0 trouvée) | Aucune imputation appliquée | Rien à imputer ; fonction générique prête et démontrée pour usage futur |
| Outliers IQR (10 signalements MINEUR) | Conservation | Variation naturelle, dans les plages plausibles ; suppression biaiserait l'analyse des sous-groupes |
| Cohérence croisée (0 incohérence) | Aucune action | Rien à corriger |

**Aucune donnée n'a été fabriquée. Aucune correction n'a été appliquée silencieusement.** Toutes les décisions sont tracées dans le journal des transformations.